In [20]:
# Save cleaning summary

cleaning_summary = {
    "raw_rows": df.shape[0],
    "raw_columns": df.shape[1],
    "cleaned_rows": df_clean.shape[0],
    "cleaned_columns": df_clean.shape[1],
    "rows_removed": df.shape[0] - df_clean.shape[0],
    "start_date": df_clean["instance_date"].min(),
    "end_date": df_clean["instance_date"].max(),
    "total_transactions": df_clean["transaction_id"].nunique(),
    "total_sales_value": df_clean["actual_worth"].sum(),
    "average_transaction_value": df_clean["actual_worth"].mean(),
    "average_price_per_sqm": df_clean["meter_sale_price"].mean()
}

cleaning_summary_df = pd.DataFrame([cleaning_summary])

summary_path = EXTERNAL_DATA_DIR / "cleaning_summary.csv"
cleaning_summary_df.to_csv(summary_path, index=False)

cleaning_summary_df

,raw_rows,raw_columns,cleaned_rows,cleaned_columns,rows_removed,start_date,end_date,total_transactions,total_sales_value,average_transaction_value,average_price_per_sqm
0,1047965,46,1044151,27,3814,1995-03-07,2023-03-17,1044151,"2,670,352,977,548.00","2,557,439.47","12,858.06"


In [19]:
# Save cleaned dataset

processed_file_path = PROCESSED_DATA_DIR / "dubai_real_estate_transactions_cleaned.csv"

df_clean.to_csv(processed_file_path, index=False)

print("Cleaned dataset saved to:", processed_file_path)

Cleaned dataset saved to: c:\Projects\dubai-real-estate-price-intelligence-dashboard\data\processed\dubai_real_estate_transactions_cleaned.csv


In [18]:
# Final clean dataset summary

print("Final dataset shape:", df_clean.shape)
print("Date range:", df_clean["instance_date"].min(), "to", df_clean["instance_date"].max())
print("Total transactions:", df_clean["transaction_id"].nunique())
print("Total sales value:", df_clean["actual_worth"].sum())
print("Average transaction value:", df_clean["actual_worth"].mean())
print("Average price per sqm:", df_clean["meter_sale_price"].mean())

df_clean.head()

Final dataset shape: (1044151, 27)
Date range: 1995-03-07 00:00:00 to 2023-03-17 00:00:00
Total transactions: 1044151
Total sales value: 2670352977548.0
Average transaction value: 2557439.467613401
Average price per sqm: 12858.06306821523


,transaction_id,procedure_id,trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_id,area_name_en,building_name_en,project_name_en,master_project_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,meter_rent_price,transaction_year,transaction_month,transaction_quarter,transaction_year_month
0,1-11-2001-165,11,Sales,Sell,2001-02-24,Land,Unknown,Commercial,Existing Properties,364,Al Wasl,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,"1,393.55","1,350,000.00",968.75,NaN,"2,001.00",2.00,1.00,2001-02
1,3-9-2004-223,9,Gifts,Grant,2004-12-13,Villa,Unknown,Commercial,Existing Properties,365,Al Hudaiba,Unknown,Unknown,Unknown,Al Jafiliya Metro Station,Dubai Mall,Burj Khalifa,Unknown,0,"1,728.00","2,790,000.00","1,614.58",NaN,"2,004.00",12.00,4.00,2004-12
2,2-13-1996-119,13,Mortgages,Mortgage Registration,2001-03-12,Land,Unknown,Commercial,Existing Properties,390,Burj Khalifa,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,0,929.03,"20,000,000.00","21,527.83",NaN,"2,001.00",3.00,1.00,2001-03
3,2-14-2005-222,14,Mortgages,Modify Mortgage,2005-09-20,Building,Unknown,Residential / Commercial,Existing Properties,388,Oud Metha,Unknown,Unknown,Unknown,Oud Metha Metro Station,Dubai Mall,Dubai International Airport,Unknown,0,"2,673.28","25,000,000.00","9,351.81",NaN,"2,005.00",9.00,3.00,2005-09
4,3-9-2012-874,9,Gifts,Grant,2012-10-11,Villa,Unknown,Residential,Existing Properties,276,Al Bada,Unknown,Unknown,Unknown,Trade Centre Metro Station,Dubai Mall,Burj Khalifa,Unknown,0,"1,541.17","9,000,000.00","5,839.72",NaN,"2,012.00",10.00,4.00,2012-10


In [17]:
# Remove duplicate transaction IDs

before_rows = len(df_clean)

df_clean = df_clean.drop_duplicates(subset=["transaction_id"]).copy()

after_rows = len(df_clean)

print("Rows before duplicate removal:", before_rows)
print("Rows after duplicate removal:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before duplicate removal: 1044151
Rows after duplicate removal: 1044151
Rows removed: 0


In [16]:
# Check duplicate transaction IDs

duplicate_transaction_ids = df_clean["transaction_id"].duplicated().sum()

print("Duplicate transaction IDs:", duplicate_transaction_ids)

Duplicate transaction IDs: 0


In [15]:
# Fill missing categorical values

categorical_columns = df_clean.select_dtypes(include="object").columns

for col in categorical_columns:
    df_clean[col] = df_clean[col].fillna("Unknown")

print("Missing values after categorical filling:")
df_clean.isnull().sum().sort_values(ascending=False).head(20)

C:\Users\hanir\AppData\Local\Temp\ipykernel_25444\4005756467.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_clean.select_dtypes(include="object").columns


Missing values after categorical filling:


meter_rent_price        1009853
procedure_id                  0
transaction_id                0
procedure_name_en             0
instance_date                 0
property_type_en              0
property_sub_type_en          0
property_usage_en             0
reg_type_en                   0
area_id                       0
trans_group_en                0
area_name_en                  0
building_name_en              0
master_project_en             0
project_name_en               0
nearest_mall_en               0
nearest_landmark_en           0
rooms_en                      0
nearest_metro_en              0
has_parking                   0
dtype: int64

In [14]:
# Remove invalid numerical values

before_rows = len(df_clean)

df_clean = df_clean[
    (df_clean["actual_worth"] > 0) &
    (df_clean["procedure_area"] > 0) &
    (df_clean["meter_sale_price"] > 0)
].copy()

after_rows = len(df_clean)

print("Rows before invalid value filtering:", before_rows)
print("Rows after invalid value filtering:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before invalid value filtering: 1044155
Rows after invalid value filtering: 1044151
Rows removed: 4


In [13]:
# Remove rows with missing critical values

before_rows = len(df_clean)

critical_columns = [
    "transaction_id",
    "instance_date",
    "area_name_en",
    "property_type_en",
    "procedure_area",
    "actual_worth",
    "meter_sale_price"
]

critical_columns = [col for col in critical_columns if col in df_clean.columns]

df_clean = df_clean.dropna(subset=critical_columns)

after_rows = len(df_clean)

print("Rows before cleaning:", before_rows)
print("Rows after removing missing critical values:", after_rows)
print("Rows removed:", before_rows - after_rows)

Rows before cleaning: 1047965
Rows after removing missing critical values: 1044155
Rows removed: 3810


In [12]:
# Create date-based features

df_clean["transaction_year"] = df_clean["instance_date"].dt.year
df_clean["transaction_month"] = df_clean["instance_date"].dt.month
df_clean["transaction_quarter"] = df_clean["instance_date"].dt.quarter
df_clean["transaction_year_month"] = df_clean["instance_date"].dt.to_period("M").astype(str)

df_clean[[
    "instance_date",
    "transaction_year",
    "transaction_month",
    "transaction_quarter",
    "transaction_year_month"
]].head()

,instance_date,transaction_year,transaction_month,transaction_quarter,transaction_year_month
0,2001-02-24,"2,001.00",2.00,1.00,2001-02
1,2004-12-13,"2,004.00",12.00,4.00,2004-12
2,2001-03-12,"2,001.00",3.00,1.00,2001-03
3,2005-09-20,"2,005.00",9.00,3.00,2005-09
4,2012-10-11,"2,012.00",10.00,4.00,2012-10


In [11]:
# Convert instance_date to datetime

df_clean["instance_date"] = pd.to_datetime(df_clean["instance_date"], errors="coerce")

print("Missing dates after conversion:", df_clean["instance_date"].isnull().sum())

df_clean[["instance_date"]].head()

Missing dates after conversion: 5


C:\Users\hanir\AppData\Local\Temp\ipykernel_25444\3531200435.py:3: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_clean["instance_date"] = pd.to_datetime(df_clean["instance_date"], errors="coerce")


,instance_date
0,2001-02-24
1,2004-12-13
2,2001-03-12
3,2005-09-20
4,2012-10-11


In [10]:
# Select useful columns for analysis

selected_columns = [
    "transaction_id",
    "procedure_id",
    "trans_group_en",
    "procedure_name_en",
    "instance_date",
    "property_type_en",
    "property_sub_type_en",
    "property_usage_en",
    "reg_type_en",
    "area_id",
    "area_name_en",
    "building_name_en",
    "project_name_en",
    "master_project_en",
    "nearest_metro_en",
    "nearest_mall_en",
    "nearest_landmark_en",
    "rooms_en",
    "has_parking",
    "procedure_area",
    "actual_worth",
    "meter_sale_price",
    "meter_rent_price"
]

selected_columns = [col for col in selected_columns if col in df_clean.columns]

df_clean = df_clean[selected_columns].copy()

print("Selected columns:", len(df_clean.columns))
print("Shape after selecting columns:", df_clean.shape)

df_clean.head()

Selected columns: 23
Shape after selecting columns: (1047965, 23)


,transaction_id,procedure_id,trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_id,area_name_en,building_name_en,project_name_en,master_project_en,nearest_metro_en,nearest_mall_en,nearest_landmark_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,meter_rent_price
0,1-11-2001-165,11,Sales,Sell,24-02-2001,Land,NaN,Commercial,Existing Properties,364,Al Wasl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,"1,393.55","1,350,000.00",968.75,NaN
1,3-9-2004-223,9,Gifts,Grant,13-12-2004,Villa,NaN,Commercial,Existing Properties,365,Al Hudaiba,NaN,NaN,NaN,Al Jafiliya Metro Station,Dubai Mall,Burj Khalifa,NaN,0,"1,728.00","2,790,000.00","1,614.58",NaN
2,2-13-1996-119,13,Mortgages,Mortgage Registration,12-03-2001,Land,NaN,Commercial,Existing Properties,390,Burj Khalifa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,929.03,"20,000,000.00","21,527.83",NaN
3,2-14-2005-222,14,Mortgages,Modify Mortgage,20-09-2005,Building,NaN,Residential / Commercial,Existing Properties,388,Oud Metha,NaN,NaN,NaN,Oud Metha Metro Station,Dubai Mall,Dubai International Airport,NaN,0,"2,673.28","25,000,000.00","9,351.81",NaN
4,3-9-2012-874,9,Gifts,Grant,11-10-2012,Villa,NaN,Residential,Existing Properties,276,Al Bada,NaN,NaN,NaN,Trade Centre Metro Station,Dubai Mall,Burj Khalifa,NaN,0,"1,541.17","9,000,000.00","5,839.72",NaN


In [9]:
# Unique values in important categorical columns

category_cols = [
    "property_type_en",
    "property_sub_type_en",
    "property_usage_en",
    "reg_type_en",
    "area_name_en",
    "rooms_en"
]

available_category_cols = [col for col in category_cols if col in df_clean.columns]

for col in available_category_cols:
    print("\n", "="*50)
    print(col)
    print("Unique values:", df_clean[col].nunique())
    print(df_clean[col].dropna().unique()[:20])


property_type_en
Unique values: 4
<ArrowStringArray>
['Land', 'Villa', 'Building', 'Unit']
Length: 4, dtype: str

property_sub_type_en
Unique values: 17
<ArrowStringArray>
[             'Villa',           'Building',               'Flat',
               'Shop',             'Office',    'Hotel Apartment',
           'Workshop',        'Hotel Rooms', 'Stacked Townhouses',
          'Warehouse',    'Sized Partition',             'Clinic',
              'Store',          'Gymnasium',              'Hotel',
         'Show Rooms',            'Parking']
Length: 17, dtype: str

property_usage_en
Unique values: 11
<ArrowStringArray>
[                           'Commercial',
              'Residential / Commercial',
                           'Residential',
                                 'Other',
                            'Industrial',
                          'Agricultural',
                             'Multi-Use',
               'Industrial / Commercial',
                          'Hospi

In [8]:
# Basic statistics for important numerical columns

numeric_check_cols = [
    "actual_worth",
    "meter_sale_price",
    "procedure_area"
]

available_numeric_cols = [col for col in numeric_check_cols if col in df_clean.columns]

df_clean[available_numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
actual_worth,"1,044,160.00","2,557,483.16","5,968,448.31",1.00,"680,400.00","1,245,500.00","2,285,490.25","99,971,250.00"
meter_sale_price,"1,047,965.00","13,488.75","124,693.60",0.00,"6,210.91","9,334.01","14,047.38","34,995,777.30"
procedure_area,"1,047,965.00","1,599.63","355,851.27",0.05,72.28,122.47,255.57,"342,103,430.80"


In [7]:
# Missing values in important columns

df_clean[available_columns].isnull().sum().sort_values(ascending=False)

project_name_en         390593
building_name_en        326895
rooms_en                248935
nearest_mall_en         242142
property_sub_type_en    240146
nearest_metro_en        236824
nearest_landmark_en     125724
actual_worth              3805
instance_date                5
transaction_id               0
area_name_en                 0
property_type_en             0
property_usage_en            0
reg_type_en                  0
meter_sale_price             0
procedure_area               0
has_parking                  0
dtype: int64

In [6]:
# Preview important columns

important_columns = [
    "transaction_id",
    "instance_date",
    "property_type_en",
    "property_sub_type_en",
    "property_usage_en",
    "reg_type_en",
    "area_name_en",
    "building_name_en",
    "project_name_en",
    "actual_worth",
    "meter_sale_price",
    "procedure_area",
    "rooms_en",
    "has_parking",
    "nearest_metro_en",
    "nearest_mall_en",
    "nearest_landmark_en"
]

available_columns = [col for col in important_columns if col in df_clean.columns]

df_clean[available_columns].head()

,transaction_id,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,building_name_en,project_name_en,actual_worth,meter_sale_price,procedure_area,rooms_en,has_parking,nearest_metro_en,nearest_mall_en,nearest_landmark_en
0,1-11-2001-165,24-02-2001,Land,NaN,Commercial,Existing Properties,Al Wasl,NaN,NaN,"1,350,000.00",968.75,"1,393.55",NaN,0,NaN,NaN,NaN
1,3-9-2004-223,13-12-2004,Villa,NaN,Commercial,Existing Properties,Al Hudaiba,NaN,NaN,"2,790,000.00","1,614.58","1,728.00",NaN,0,Al Jafiliya Metro Station,Dubai Mall,Burj Khalifa
2,2-13-1996-119,12-03-2001,Land,NaN,Commercial,Existing Properties,Burj Khalifa,NaN,NaN,"20,000,000.00","21,527.83",929.03,NaN,0,NaN,NaN,NaN
3,2-14-2005-222,20-09-2005,Building,NaN,Residential / Commercial,Existing Properties,Oud Metha,NaN,NaN,"25,000,000.00","9,351.81","2,673.28",NaN,0,Oud Metha Metro Station,Dubai Mall,Dubai International Airport
4,3-9-2012-874,11-10-2012,Villa,NaN,Residential,Existing Properties,Al Bada,NaN,NaN,"9,000,000.00","5,839.72","1,541.17",NaN,0,Trade Centre Metro Station,Dubai Mall,Burj Khalifa


In [5]:
# Show all columns with index numbers

for i, col in enumerate(df_clean.columns):
    print(i, ":", col)

0 : transaction_id
1 : procedure_id
2 : trans_group_id
3 : trans_group_ar
4 : trans_group_en
5 : procedure_name_ar
6 : procedure_name_en
7 : instance_date
8 : property_type_id
9 : property_type_ar
10 : property_type_en
11 : property_sub_type_id
12 : property_sub_type_ar
13 : property_sub_type_en
14 : property_usage_ar
15 : property_usage_en
16 : reg_type_id
17 : reg_type_ar
18 : reg_type_en
19 : area_id
20 : area_name_ar
21 : area_name_en
22 : building_name_ar
23 : building_name_en
24 : project_number
25 : project_name_ar
26 : project_name_en
27 : master_project_en
28 : master_project_ar
29 : nearest_landmark_ar
30 : nearest_landmark_en
31 : nearest_metro_ar
32 : nearest_metro_en
33 : nearest_mall_ar
34 : nearest_mall_en
35 : rooms_ar
36 : rooms_en
37 : has_parking
38 : procedure_area
39 : actual_worth
40 : meter_sale_price
41 : rent_value
42 : meter_rent_price
43 : no_of_parties_role_1
44 : no_of_parties_role_2
45 : no_of_parties_role_3


In [4]:
# Check cleaned columns

for col in df_clean.columns:
    print(col)

transaction_id
procedure_id
trans_group_id
trans_group_ar
trans_group_en
procedure_name_ar
procedure_name_en
instance_date
property_type_id
property_type_ar
property_type_en
property_sub_type_id
property_sub_type_ar
property_sub_type_en
property_usage_ar
property_usage_en
reg_type_id
reg_type_ar
reg_type_en
area_id
area_name_ar
area_name_en
building_name_ar
building_name_en
project_number
project_name_ar
project_name_en
master_project_en
master_project_ar
nearest_landmark_ar
nearest_landmark_en
nearest_metro_ar
nearest_metro_en
nearest_mall_ar
nearest_mall_en
rooms_ar
rooms_en
has_parking
procedure_area
actual_worth
meter_sale_price
rent_value
meter_rent_price
no_of_parties_role_1
no_of_parties_role_2
no_of_parties_role_3


In [3]:
# Clean column names

df_clean = df.copy()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
    .str.replace("/", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

print("Column names cleaned.")

df_clean.columns.tolist()

Column names cleaned.


['transaction_id',
 'procedure_id',
 'trans_group_id',
 'trans_group_ar',
 'trans_group_en',
 'procedure_name_ar',
 'procedure_name_en',
 'instance_date',
 'property_type_id',
 'property_type_ar',
 'property_type_en',
 'property_sub_type_id',
 'property_sub_type_ar',
 'property_sub_type_en',
 'property_usage_ar',
 'property_usage_en',
 'reg_type_id',
 'reg_type_ar',
 'reg_type_en',
 'area_id',
 'area_name_ar',
 'area_name_en',
 'building_name_ar',
 'building_name_en',
 'project_number',
 'project_name_ar',
 'project_name_en',
 'master_project_en',
 'master_project_ar',
 'nearest_landmark_ar',
 'nearest_landmark_en',
 'nearest_metro_ar',
 'nearest_metro_en',
 'nearest_mall_ar',
 'nearest_mall_en',
 'rooms_ar',
 'rooms_en',
 'has_parking',
 'procedure_area',
 'actual_worth',
 'meter_sale_price',
 'rent_value',
 'meter_rent_price',
 'no_of_parties_role_1',
 'no_of_parties_role_2',
 'no_of_parties_role_3']

In [2]:
# Load raw dataset

file_path = RAW_DATA_DIR / "Transactions.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (1047965, 46)


,transaction_id,procedure_id,trans_group_id,trans_group_ar,trans_group_en,procedure_name_ar,procedure_name_en,instance_date,property_type_id,property_type_ar,property_type_en,property_sub_type_id,property_sub_type_ar,property_sub_type_en,property_usage_ar,property_usage_en,reg_type_id,reg_type_ar,reg_type_en,area_id,area_name_ar,area_name_en,building_name_ar,building_name_en,project_number,project_name_ar,project_name_en,master_project_en,master_project_ar,nearest_landmark_ar,nearest_landmark_en,nearest_metro_ar,nearest_metro_en,nearest_mall_ar,nearest_mall_en,rooms_ar,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
0,1-11-2001-165,11,1,مبايعات,Sales,بيع,Sell,24-02-2001,1,أرض,Land,NaN,NaN,NaN,تجاري,Commercial,1,العقارات القائمة,Existing Properties,364,الوصل,Al Wasl,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,"1,393.55","1,350,000.00",968.75,NaN,NaN,1.00,1.00,0.00
1,3-9-2004-223,9,3,هبات,Gifts,هبه,Grant,13-12-2004,4,فيلا,Villa,NaN,NaN,NaN,تجاري,Commercial,1,العقارات القائمة,Existing Properties,365,الحضيبه,Al Hudaiba,NaN,NaN,NaN,NaN,NaN,NaN,NaN,برج خليفة,Burj Khalifa,محطة مترو الجافلية,Al Jafiliya Metro Station,مول دبي,Dubai Mall,NaN,NaN,0,"1,728.00","2,790,000.00","1,614.58",NaN,NaN,1.00,1.00,0.00
2,2-13-1996-119,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,12-03-2001,1,أرض,Land,NaN,NaN,NaN,تجاري,Commercial,1,العقارات القائمة,Existing Properties,390,برج خليفة,Burj Khalifa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,929.03,"20,000,000.00","21,527.83",NaN,NaN,1.00,1.00,0.00
3,2-14-2005-222,14,2,رهون,Mortgages,تعديل رهن,Modify Mortgage,20-09-2005,2,مبنى,Building,NaN,NaN,NaN,سكني / تجاري,Residential / Commercial,1,العقارات القائمة,Existing Properties,388,عود ميثا,Oud Metha,NaN,NaN,NaN,NaN,NaN,NaN,NaN,مطار دبي الدولي,Dubai International Airport,محطة مترو عود ميثاء,Oud Metha Metro Station,مول دبي,Dubai Mall,NaN,NaN,0,"2,673.28","25,000,000.00","9,351.81",NaN,NaN,1.00,1.00,0.00
4,3-9-2012-874,9,3,هبات,Gifts,هبه,Grant,11-10-2012,4,فيلا,Villa,NaN,NaN,NaN,سكني,Residential,1,العقارات القائمة,Existing Properties,276,البدع,Al Bada,NaN,NaN,NaN,NaN,NaN,NaN,NaN,برج خليفة,Burj Khalifa,محطة مترو المركز التجاري,Trade Centre Metro Station,مول دبي,Dubai Mall,NaN,NaN,0,"1,541.17","9,000,000.00","5,839.72",NaN,NaN,1.00,1.00,0.00


In [1]:
# Dubai Real Estate Price Intelligence Dashboard
# Notebook 02: Data Cleaning

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EXTERNAL_DATA_DIR = PROJECT_ROOT / "data" / "external"

print("Setup completed.")
print("Project root:", PROJECT_ROOT)

Setup completed.
Project root: c:\Projects\dubai-real-estate-price-intelligence-dashboard
